1.LLM 기초와 금융 텍스트 분석 입문

In [ ]:
import sys, os

print('Python 버전:', sys.version.split()[0])
print('현재 디렉토리:', os.getcwd())

#OpenAI 라이브러리 설치 및 키 설정
!pip install openai -q

from openai import OpenAI

#API Key 설정
os.environ['OPENAI_API_KEY'] = ''

#OpenAI 클라이언트 생성
client = OpenAI()

print('API 준비 완료')

Python 버전: 3.12.13
현재 디렉토리: /content
API 준비 완료


[실습 2] 금융 뉴스 한 문장을 LLM으로 한 줄 요약

In [24]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'))

news = '''A전자, 3분기 영업이익 10조원 발표.
메모리 반도체 수요 회복이 실적 개선을 이끌었다.'''

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user',
               'content':f'한 문장 요약:{news}'}])

print(resp.choices[0].message.content)

A전자가 3분기 영업이익 10조원을 발표했으며, 이는 메모리 반도체 수요 회복에 기인한 실적 개선의 결과다.


1.2 LLM 구조 개념 이해

In [ ]:
# 1. 환경변수에 API 키 저장
import os

os.environ['OPENAI_API_KEY'] = ''

In [5]:
from openai import OpenAI; import numpy as np
import os # Import os to access environment variables

# Initialize the client by explicitly passing the API key from the environment variable
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def emb(t):
  r = client.embeddings.create(
      model='text-embedding-3-small', input=t)
  return np.array(r.data[0].embedding)

v = emb("금리가 인상되면 채권 가격은 하락할 수 있다.")

print(v)
print(type(v))
print(v.shape)

[ 0.01661682  0.02043152 -0.03089905 ... -0.00103283  0.00326347
  0.00203705]
<class 'numpy.ndarray'>
(1536,)


[실습 1] 금융 단어 3개('채권','배당','환율')의 임베딩을 구해 '금리'와의 유사도를 비교하시오.

In [12]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def emb(t):
  r = client.embeddings.create(
      model='text-embedding-3-small',input=t)
  return np.array(r.data[0].embedding)

def sim(a,b): #두 Embedding 벡터간 유사도를 계산하는 함수
 return a @ b / (np.linalg.norm(a)*np.linalg.norm(b))
 #두 문장의 의미의 유사도를 0~1 사이의 숫자로 계산하는 함수, 코사인 유사도

base = emb('금리')
for w in ['채권','배당','환율']:
  print(w, round(sim(base, emb(w)), 3))

채권 0.297
배당 0.273
환율 0.343


[실습 2] 금융 문장을 토큰 단위로 나눠 토큰 개수를 세기

In [15]:
!pip install tiktoken -q

import tiktoken

enc = tiktoken.get_encoding('cl100k_base')
text = '삼성전자 3분기 영업이익 어닝 서프라이즈'
tokens = enc.encode(text)

print('토큰 개수:', len(tokens))
print('토큰 번호(ID):', tokens[:8])

# 토큰 확인하기
for token in tokens:
  print(f"ID: {token:6d}"
        f"-{enc.decode([token])!r}"
  )

토큰 개수: 27
토큰 번호(ID): [80690, 120, 33931, 66965, 26799, 220, 18, 80816]
ID:  80690-'�'
ID:    120-'�'
ID:  33931-'성'
ID:  66965-'전'
ID:  26799-'자'
ID:    220-' '
ID:     18-'3'
ID:  80816-'분'
ID:  21121-'기'
ID:  39623-' �'
ID:    223-'�'
ID:  13879-'�'
ID:    227-'�'
ID:  13094-'이'
ID:   6026-'�'
ID:    113-'�'
ID:  80402-' �'
ID:    112-'�'
ID:   9019-'�'
ID:    251-'�'
ID:  90960-' 서'
ID:    169-'�'
ID:  63644-'��'
ID:  51440-'라'
ID:  13094-'이'
ID:  96064-'�'
ID:    230-'�'


[실습 3] 여러 단어 중 '주가'와 가장 가까운 단어를 임베딩으로 찾기

In [19]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def emb(t):
  r = client.embeddings.create(
      model='text-embedding-3-small',input=t)
  return np.array(r.data[0].embedding)

def sim(a,b): #두 Embedding 벡터간 유사도를 계산하는 함수
 return a @ b / (np.linalg.norm(a)*np.linalg.norm(b))
 #두 문장의 의미의 유사도를 0~1 사이의 숫자로 계산하는 함수, 코사인 유사도

base = emb('주가')

words = ['시세', '날씨','배당','점심']

best = max(words, key=lambda w: sim(base, emb(w)))

print('가장 가까운 단어:', best)

가장 가까운 단어: 점심


In [23]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def emb(t):
  r = client.embeddings.create(
      model='text-embedding-3-small',input=t)
  return np.array(r.data[0].embedding)

def sim(a,b):
 return a @ b / (np.linalg.norm(a)*np.linalg.norm(b))

base = emb('주가')

words = [
    "주식의 현재 거래 가격과 시세",
    "오늘 비가 오고 기온이 내려가는 날씨",
    "기업이 주주에게 지급하는 배당금",
    "점심시간에 식사를 하는 것"
]

print('가장 가까운 단어:', best)

for w in words:
  score = sim(base, emb(w))
  print(w, round(score, 3))

가장 가까운 단어: 기업이 주주에게 지급하는 배당금
주식의 현재 거래 가격과 시세 0.2
오늘 비가 오고 기온이 내려가는 날씨 0.242
기업이 주주에게 지급하는 배당금 0.287
점심시간에 식사를 하는 것 0.21


좋은 프롬프트의 예시

In [24]:
# 역할 + 형식을 지정한 프롬프트 : # [함수] create(): 프롬프트를 보내 답변을 생성하는 함수

prompt = '''너는 금융 애널리스트다.
다음 뉴스의 투자 감성을 긍정/부정/중립 중 하나로만 답하라.
뉴스: A기업 3분기 영업이익 40% 감소'''

r = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user','content':prompt}])

print(r.choices[0].message.content)

부정


*   Zero-shot vs Few-shot(예시를 함께 줌)
*   Temperature(자유도 조절로 창의성 부여)

[실습 1] Zero-shot과 Few-shot 프롬프트로 같은 뉴스를 분류

In [29]:
news = 'A기업, 3분기 영업이익 전년比 40% 감소'

#Zero-shot
p1 = f'다음 뉴스의 투자 감성을 분류: {news}'

#Few-shot (예시 제공)
p2 = f'''예시) 실적 급증->긍정 / 적자 전환->부정
다음을 긍정/부정/중립 중 하나로만 답하라:{news}'''

for name, p in [('Zero-shot', p1), ('Few-shot', p2)]:
  r = client.chat.completions.create(
      model='gpt-4o-mini',
      messages=[{'role':'user','content':p}])

  result = r.choices[0].message.content
  print(f'{name}: {result}')

Zero-shot: 해당 뉴스는 A기업의 3분기 영업이익이 전년 대비 40% 감소했다고 전하고 있습니다. 이는 기업의 실적 악화를 의미하며, 대체로 부정적인 투자 감성을 나타냅니다. 투자자들은 이 정보를 바탕으로 A기업의 미래 성장 가능성이나 수익성에 대해 우려할 수 있기 때문에, 전반적으로 부정적인 감성으로 분류할 수 있습니다.
Few-shot: 부정


[실습 2] 출력 형식을 JSON으로 지정해 뉴스에서 핵심 정보를 추출

In [51]:
import json


report = 'B전자 3분기 매출 70조(+12%), 영업이익 10조(+25%)'

prompt = f'''다음에서 핵심 정보를 JSON으로만 추출하라.
키: 매출, 영업이익, 증감률
{report}'''

r = client.chat.completions.create(
    model='gpt-4o-mini'
    ,messages=[{'role':'user','content':prompt}]
    ,response_format={'type':'json_object'})

result_text = r.choices[0].message.content

# JSON 문자열을 파이썬의 딕셔너리로 변환
result = json.loads(result_text)

print(result)

# JSON 파일로 현재 위치에 저장
with open('/content/drive/MyDrive/LLM기반/report.json', 'w', encoding='utf-8') as f:
  json.dump(result, f, ensure_ascii=False, indent=4)

{'매출': {'금액': '70조', '증감률': '+12%'}, '영업이익': {'금액': '10조', '증감률': '+25%'}}


In [30]:
report = 'B전자 3분기 매출 70조(+12%), 영업이익 10조(+25%)'

prompt = f"""다음 실적 보고서를 읽고, 보기 쉬운 표(Markdown Table) 형식으로 정리해줘.
{report}"""

r = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}]
)

print(r.choices[0].message.content)

아래는 B전자의 3분기 실적 보고서를 정리한 표입니다.

| 항목       | 수치        | 변동률  |
|------------|------------|--------|
| 매출       | 70조 원    | +12%   |
| 영업이익   | 10조 원    | +25%   |


[실습 3] Temperature를 바꿔 같은 질문의 답이 어떻게 달라지는지 확인

In [47]:
news = '''미국·이란 전쟁 재격화와 홍해 물류 차질로 국제 유가가 급등하여,
        브렌트유는 배럴당 100달러를 넘었고,
        서부텍사스산원유(WTI)는 92달러선을 기록'''
prompt = f'이 뉴스로 투자자용 코멘트를 1줄 작성:{news}'

for temp in [0.0, 0.5, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temp)
    print(f'[temp={temp}]', r.choices[0].message.content)

[temp=0.0] "미국과 이란 간의 긴장 재격화 및 홍해 물류 차질로 인해 국제 유가가 급등, 브렌트유가 100달러를 초과하며 에너지 섹터에 대한 투자 기회를 주목해야 할 시점입니다."
[temp=0.5] "미국과 이란 간의 긴장 고조 및 홍해 물류 차질로 국제 유가가 급등하며, 투자자들은 에너지 섹터의 변동성을 주의 깊게 살펴야 할 시점입니다."
[temp=1.0] "미국과 이란 간의 갈등 재점화 및 홍해 물류 장애로 국제 유가가 급등하며, 브렌트유가 100달러를 초과하고 WTI가 92달러에 근접, 향후 에너지 시장의 변동성이 더욱 커질 것으로 예상됩니다."


1.4 금융 문서 요약 & 핵심 정보 추출

[실습 2] 리포트에서 실적 · 전망 · 리스크를 JSON으로 구조화 추출

In [50]:
report = '''C바이오 3분기 적자 지속(영업손실 200억).
다만 신약 임상3상 성공으로 내년 흑자 전환 기대.
단, FDA 승인 지연 시 실적 불확실성 존재.'''

prompt = f'''다음에서 JSON으로 추출하라.
키: 실적, 전망, 리스크
{report}'''

r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'})

# JSON 문자열 추출
result_text = r.choices[0].message.content

# result_text -> JSON 파일로 변환하기
result = json.loads(result_text)

# JSON 파일로 저장하기
with open('/content/drive/MyDrive/LLM기반/report2.json', 'w', encoding='utf-8') as f:
  json.dump(result, f, ensure_ascii=False, indent=4)

print('저장 성공')

저장 성공


저장한 파일 다운로드하기

In [52]:
from google.colab import files

files.download('/content/drive/MyDrive/LLM기반/report2.json')

print('다운로드 완료')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드 완료


[실습 3] 존재하지 않는 정보를 물어 '환각' 발생 확인

In [57]:
from openai import OpenAI; client = OpenAI()

#일부러 자료에 없는 내용을 질문
report = 'B전자 3분기 매출 70조, 영업이익 10조'
q = f'''위 자료에서 D전자의 4분기 순이익은? 자료: {report}
자료에 없으면 반드시 '알 수 없음'이라고 답함'''

r = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user','content':q}])

print(r.choices[0].message.content)

알 수 없음


[미니 프로젝트] 금융 뉴스 자동 분석기



*   처리 단계: 뉴스 입력 -> 요약 -> 감성 분류 -> 핵심정보 추출 -> 결과 누적
*   LLM 호출: 프롬프트에 역할과 형식(JSON)을 지정해 일관된 결과 확보
*   구조화: pandas DataFrame으로 결과를 표 형태로 누적, 집계





In [64]:
import pandas as pd, json
from openai import OpenAI
!pip install PyPDF2 -q # Install the missing library
import PyPDF2  # PDF에서 텍스트를 추출하는 라이브러리 (pip install PyPDF2)


client = OpenAI()

def extract_pdf_text(path):
    """PDF 파일 경로 -> 전체 텍스트 문자열"""
    text = ''
    with open(path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ''
    return text

def analyze(report):
    """리포트 1건 -> {요약, 감성, 핵심정보} JSON"""
    prompt = f'''당신은 보험업에 종사하는 전문가이다.
아래 리포트를 읽고 다음 JSON 형식으로만 답하라. 다른 텍스트는 절대 포함하지 마라.
{{
  "summary": "리포트 내용을 한 줄로 요약",
  "sentiment": "긍정/부정/중립 중 하나",
  "event": "리포트에서 도출한 핵심 수치나 이벤트"
}}

리포트:
{report}'''

    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'})
    return json.loads(r.choices[0].message.content)

# 분석할 리포트 목록
repo_list = [
    '/content/drive/MyDrive/LLM기반/2026년 보험산업 전망.pdf',
    '/content/drive/MyDrive/LLM기반/보험업의 AI 실행 격차.pdf',
    '/content/drive/MyDrive/LLM기반/생명보험회사의 요양사업 진출 현황과 과제.pdf'
]

# 1) 리포트별로 텍스트 추출 -> 분석 -> 결과 누적
rows = []
for path in repo_list:
    text = extract_pdf_text(path)
    result = analyze(text)
    result['파일명'] = path.split('/')[-1]
    rows.append(result)

df = pd.DataFrame(rows)
print(df)

# 2) JSON 파일로 현재 위치에 저장
df.to_json('/content/drive/MyDrive/LLM기반/report3.json',
           orient='records', force_ascii=False, indent=4)

                                             summary sentiment  \
0                        2026년 보험산업 전망과 주요 과제에 대한 분석        부정   
1  보험업에서의 AI 활용은 잠재적 가능성이 크지만, 실제 활용은 저조하여 실행 격차가...        부정   
2  초고령 사회 진입으로 생명보험회사의 요양사업 진출이 증가하고 있으며, 이는 양질의 ...        부정   

                                               event  \
0                       2026년 보험산업 수입보험료 성장률 2.3% 전망   
1  보험 관련 직업의 AI 잠재노출도는 0.49~0.54, 관측노출도는 0.054~0....   
2  장기요양등급 판정자가 2015년 595,974명에서 2024년 1,252,649명으...   

                          파일명  
0           2026년 보험산업 전망.pdf  
1           보험업의 AI 실행 격차.pdf  
2  생명보험회사의 요양사업 진출 현황과 과제.pdf  


In [65]:
import pandas as pd, json
from openai import OpenAI

client = OpenAI()

# 뉴스 1건 -> 요약(summary)+감성(sentiment)+이벤트(event)분석함수
def analyze(news):

  # JSON 형식으로 3가지를 한 번에 요청
  prompt = f'''다음 뉴스를 분석하여 JSON으로 출력하라.
  summary: 한 줄 요약
  sentiment: 긍정/부정/중립 중 하나
  event: 이벤트 유형(실적발표/신제품출시/M&A 등)
  뉴스: {news}'''

  # LLM 호출
  r = client.chat.completions.create(
      model='gpt-4o-mini',
      messages=[{'role':'user','content':prompt}],
      response_format={'type':'json_object'})

  return json.loads(r.choices[0].message.content) # dict 반환

# 여러 뉴스를 반복하여 분석 후 표로 정리

news_list = ['비누칠에 때밀이까지...청계천에 나타난 "목욕 빌런"',
             '미 이란 무력 충돌 소강국면...협상 기대 속 전면전 우려는 여전',
             '경기도일자리재단, 무료 취업 지원 서비스 제공',
             'SK하이닉스 ADR 프리미엄 과열']

df = pd.DataFrame(analyze(n) for n in news_list)

df

,summary,sentiment,event,뉴스
0,청계천에서 비누칠과 때밀이 등 목욕 문화가 확산되고 있다.,중립,사회적 이슈,비누칠에 때밀이까지...청계천에 나타난 '목욕 빌런'
1,"미 이란 무력 충돌이 일시적으로 소강상태에 있지만, 전면전에 대한 우려가 여전히 존...",중립,정세변화,미 이란 무력 충돌 소강국면...협상 기대 속 전면전 우려는 여전
2,경기도일자리재단이 무료 취업 지원 서비스를 제공하기로 했다.,긍정,서비스 제공,NaN
3,SK하이닉스의 ADR 프리미엄이 과열된 상황이다.,부정,주가변동,SK하이닉스 ADR 프리미엄 과열
